# S17 · Monte Carlo VaR, a stress test and a backtest

We only have 739 real days, and just 37 of them in the worst 5%. That is a thin basis
for a number a bank holds money against. **Monte Carlo simulation** fixes the shortage:
make 100,000 realistic made-up days that behave like the real ones, then read VaR and
CVaR off that much bigger pile.

Then we check the numbers against reality twice: a **stress test** (what if the COVID
crash day came back?) and a **backtest** (did our VaR keep its promise?). We finish with
a one-screen risk report for our ₹10 lakh.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to "quantile" or "the tail"? Open the primer `primers/quantiles_and_tails.md`
  for a ten-minute, picture-first version.
- Already confident with code or with probability? Look for the cells marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook downloads stock prices with yfinance, which Colab does not
# always ship. numpy, pandas and matplotlib are already there.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — get our tools ready

NumPy does maths on long lists of numbers, pandas holds tables of data, and
matplotlib draws charts.

In [ ]:
from pathlib import Path
import numpy as np                 # fast maths on lists of numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts

print("Tools ready.")

## Step 2 — download the prices

Our portfolio is the basket from S16: five well-known NSE stocks. We download three
years of daily closing prices, from January 2023 to December 2025. The dates are
fixed, so you get the same numbers as the slides.

If there is no internet (say, in an exam hall), the cell loads a saved copy of exactly
the same prices instead, so the notebook always runs. It prints which one happened.

In [ ]:
# The five stocks. The '.NS' ending means the National Stock Exchange of India.
tickers = ["TCS.NS", "INFY.NS", "RELIANCE.NS", "ITC.NS", "HDFCBANK.NS"]

# Where the saved copy lives: next to this notebook, or on the course website.
local_copy = Path("../data/nse_five_stocks_2023_2025.csv")
website_copy = ("https://girishmkulkarni.github.io/applied-maths-in-industry-site/"
                "sessions/S17/data/nse_five_stocks_2023_2025.csv")

try:
    import yfinance as yf

    # Daily prices for all five stocks at once. We keep the closing price.
    prices = yf.download(tickers, start="2023-01-01", end="2026-01-01",
                         auto_adjust=True, progress=False)["Close"]

    # Keep our column order and drop any day with a missing price.
    prices = prices[tickers].dropna()

    # An empty table means the download quietly failed.
    if len(prices) < 100:
        raise ValueError("the download came back empty")
    print("Downloaded live prices from Yahoo Finance.")

except Exception as problem:
    print("Could not download (", problem, ") - loading the saved copy instead.")
    if local_copy.exists():
        prices = pd.read_csv(local_copy, index_col=0, parse_dates=True)
    else:
        prices = pd.read_csv(website_copy, index_col=0, parse_dates=True)

print("price table:", prices.shape[0], "days,", prices.shape[1], "stocks")
prices.head()

## Step 3 — turn prices into rupee losses

Three small moves:

1. A **return** is the percentage change in price from one day to the next.
   `pct_change()` works it out for every stock and every day.
2. We put ₹2 lakh in each stock, so each has a **weight** of 0.2 (one fifth of the
   money). `returns @ weights` mixes the five returns into one portfolio return per day,
   the same `@` you met in S16.
3. Risk is about money we *lose*, so we flip the sign and multiply by the ₹10 lakh
   invested. A 1% fall becomes a loss of ₹10,000. From now on, a bigger number is worse.

In [ ]:
# Daily return of each stock: (today's price - yesterday's) / yesterday's.
returns = prices.pct_change().dropna()

# Rs 2 lakh in each of the five stocks, out of Rs 10 lakh.
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
investment = 1_000_000

# One portfolio return per day, flipped into a rupee LOSS.
# A negative return (a fall) becomes a positive loss.
losses = -(returns @ weights) * investment

print("days of data:", len(losses))
print("worst day   :", round(losses.max()), "rupees lost on", losses.idxmax().date())
print("best day    :", round(-losses.min()), "rupees gained on", losses.idxmin().date())

## Step 4 — the historical and parametric VaR, for comparison

The two answers from notebook 1, so we have something to compare the simulation with.

In [ ]:
var_hist = np.percentile(losses, 95)
var_param = losses.mean() + 1.645 * losses.std()
cvar_hist = losses[losses >= var_hist].mean()

print("Historical 95% VaR:", round(var_hist), "rupees")
print("Parametric 95% VaR:", round(var_param), "rupees")
print("Historical 95% CVaR:", round(cvar_hist), "rupees")

## Step 5 — describe how the stocks behave

To make realistic days we need two things from the real data:

- the **average** daily return of each stock, and
- the **covariance** table from S16, which records how much each stock swings and how
  each pair moves together.

The correlation table below is the easier-to-read version of the same information: 1
means two stocks move in lockstep, 0 means they are unrelated.

In [ ]:
mean_returns = returns.mean()      # one average per stock
cov_matrix = returns.cov()         # how the stocks swing and move together

print("How the five stocks move together (correlation):")
returns.corr().round(2)

## Step 6 — roll 100,000 made-up days

`np.random.multivariate_normal` draws random days from a bell curve that has our
averages and our covariance table. If two stocks tend to move together in real life,
they move together in the made-up days too. Each row of the result is one made-up day:
five stock returns.

The seed makes everyone's random days identical, so your numbers match the slides. The
last two lines are a sanity check: the made-up days should swing about as much as the
real ones.

In [ ]:
np.random.seed(0)                  # same random days for everyone

sim_returns = np.random.multivariate_normal(mean_returns, cov_matrix,
                                            size=100_000)

print("made-up days:", sim_returns.shape)
print("real average daily swing     :", returns.std().mean().round(4))
print("made-up average daily swing  :", sim_returns.std(axis=0).mean().round(4))

## Step 7 — rupee losses, then Monte Carlo VaR and CVaR

Nothing new here. The first line is the same loss formula as Step 3, on made-up days.
The VaR and CVaR lines are the same as before, just on 100,000 days instead of 739.

In [ ]:
sim_losses = -(sim_returns @ weights) * investment

var_mc = np.percentile(sim_losses, 95)
cvar_mc = sim_losses[sim_losses >= var_mc].mean()

print("Monte Carlo 95% VaR:", round(var_mc), "rupees")
print("Parametric  95% VaR:", round(var_param), "rupees")
print("Historical  95% VaR:", round(var_hist), "rupees")
print("\nMonte Carlo 95% CVaR:", round(cvar_mc), "rupees")
print("Historical  95% CVaR:", round(cvar_hist), "rupees")

Monte Carlo lands close to the parametric answer. Why? Because we rolled the made-up days
from a **bell curve**, so the simulation inherits the bell curve's view of the world. Its
CVaR is also a little below the historical CVaR: the made-up tail is thinner than the
real one. Monte Carlo is only as good as the recipe you give it.

## Step 8 — picture: made-up days against real days

Green is the pile of 100,000 made-up days, the blue outline is our 739 real days (both
scaled to the same height). The dashed line is the Monte Carlo VaR.

In [ ]:
plt.figure(figsize=(9, 5))

bins = np.arange(-32, 33, 1)   # one-thousand-rupee bins
plt.hist(sim_losses / 1000, bins=bins, density=True, color="#2E7D32", alpha=0.45,
         label="100,000 made-up days")
plt.hist(losses / 1000, bins=bins, density=True, histtype="step", color="#1F4E79",
         linewidth=1.8, label="739 real days")
plt.axvline(var_mc / 1000, color="#C0392B", linestyle="--", linewidth=2.5,
            label="Monte Carlo 95% VaR")

plt.xlabel("loss that day, thousand rupees")
plt.ylabel("share of days")
plt.title("Same line-drawing, a much bigger pile of days")
plt.legend()
plt.show()

## Step 9 — stress test: replay 23 March 2020

VaR and CVaR describe the bad days in our data. A **stress test** asks about one extreme
day we choose, even if it is not in the data. Our data starts in 2023, so it never saw
the COVID crash.

On 23 March 2020 the Nifty fell 13% in a day. Here is how each of our five stocks moved
that day, in the same order as `tickers`. We apply those moves to today's ₹10 lakh.

In [ ]:
# How each stock actually moved on 23 March 2020:
#                    TCS     Infosys  Reliance  ITC     HDFC Bank
covid_day = np.array([-0.071, -0.100,  -0.132,   -0.121, -0.126])

stress = -(covid_day @ weights) * investment

print("COVID-day replay loss:", round(stress), "rupees")
print("that is", round(stress / var_hist, 1), "times our 95% VaR")

Nobody claims a day like that is 5% likely. The claim is only: *if that day came back,
here is the damage.* VaR describes normal bad days. Stress testing sizes crisis days.

## Step 10 — backtest: did our VaR keep its promise?

A 95% VaR promises that the loss will break it on about 5% of days. We can check that
promise honestly. Stand on each day, compute the historical VaR from the **previous 250
days only**, then see whether the next day's loss broke it.

- `losses.rolling(250).quantile(0.95)` is the 95% mark of the last 250 days, for every day.
- `.shift(1)` moves it forward a day, so each day is judged by a line drawn *before* it.

In [ ]:
# VaR drawn from the previous 250 days, for every day.
var_line = losses.rolling(250).quantile(0.95).shift(1)

# Only days that have a full 250-day history behind them can be tested.
tested = var_line.notna()
breaks = losses[tested] > var_line[tested]

print("days tested:", tested.sum())
print("times the VaR line was broken:", breaks.sum())
print("a 95% VaR expects about:", round(0.05 * tested.sum()))

Close to what a good 95% VaR promises, so the model passes. Far more breaks would mean
we are understating risk. Far fewer would mean we are too cautious and holding too much
idle money. Here is the picture: the red line is the VaR, the dots are the breaks.

In [ ]:
plt.figure(figsize=(10, 5))

test_losses = losses[tested]
plt.bar(test_losses.index, test_losses / 1000, width=1.5, color="#2E75B6", alpha=0.6)
plt.plot(var_line[tested].index, var_line[tested] / 1000, color="#C0392B",
         linewidth=2, label="VaR line (previous 250 days)")
plt.scatter(test_losses[breaks].index, test_losses[breaks] / 1000, color="#C0392B",
            zorder=5, label="days the loss broke the line")

plt.ylabel("loss that day, thousand rupees")
plt.title("Backtest: count the days the loss broke the VaR line")
plt.legend()
plt.show()

## Step 11 — the risk report

Everything for our ₹10 lakh, on one screen. A risk manager would report all of these,
never just one.

In [ ]:
print("Risk report: Rs 10 lakh, five NSE stocks, 1 day")
print("-----------------------------------------------")
print("Historical  95% VaR :", round(var_hist), "rupees")
print("Parametric  95% VaR :", round(var_param), "rupees")
print("Monte Carlo 95% VaR :", round(var_mc), "rupees")
print("Historical  95% CVaR:", round(cvar_hist), "rupees")
print("COVID-day stress    :", round(stress), "rupees")

### Stretch (optional) — how much does the answer wobble run to run?

Skip this if you are new to code. A Monte Carlo number is an estimate. Run the same
simulation with a few different seeds and see how much the VaR moves. With 100,000 days
it barely moves; try `size=1000` to watch it jump.

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    np.random.seed(seed)
    trial_returns = np.random.multivariate_normal(mean_returns, cov_matrix, size=100_000)
    trial_losses = -(trial_returns @ weights) * investment
    print("seed", seed, "  Monte Carlo VaR:", round(np.percentile(trial_losses, 95)))

### Stretch (optional) — a fatter-tailed recipe: resample the real days

Skip this if you are new to code. Our bell-curve recipe missed the fat tails. One simple
fix: instead of inventing days from a bell curve, draw 100,000 days **at random from the
739 real days** (with repeats allowed). This is called *bootstrapping*. The made-up days
now carry the real fat tails, big days and all.

In [ ]:
np.random.seed(0)

# Pick 100,000 random day numbers between 0 and 738, repeats allowed.
picked_days = np.random.randint(0, len(returns), size=100_000)
boot_returns = returns.values[picked_days]

boot_losses = -(boot_returns @ weights) * investment
boot_var = np.percentile(boot_losses, 95)
boot_cvar = boot_losses[boot_losses >= boot_var].mean()

print("Bootstrap   95% VaR :", round(boot_var), "   CVaR:", round(boot_cvar))
print("Bell-curve  95% VaR :", round(var_mc), "   CVaR:", round(cvar_mc))

## Your turn: your own risk report

1. Repeat the report at **99%** instead of 95%. You need `np.percentile(..., 99)` and the
   bell-curve multiplier **2.326**. Which numbers grow the most?
2. Choose your own weights for the five stocks (they must add up to 1). Rerun Step 3 and
   every step after it. Is your portfolio riskier or safer than equal weights?
3. Write three plain sentences, the way you would explain it to a fund's investment
   committee, saying what your numbers mean for the ₹10 lakh.

In [ ]:
# Your code here: the 99% report.

*Your three sentences here.*

## What you just did

You rolled 100,000 realistic days with **Monte Carlo** and read VaR and CVaR off them,
saw that a simulation is only as good as its recipe, **stress-tested** the portfolio
against the COVID crash day, and **backtested** the VaR to check it keeps its promise.
That is the full daily risk pipeline of a real desk, on one real portfolio.

Next session we use the same Monte Carlo engine to price options.